# contest

> Phase 19 (`specs/multiagent/phase19/`): contest credit on the clue board. `ContestLog` records, as the game runs, every contest (an honest post of a clue's true value while the board shows a value a liar posted), the honest agents that were shown the lie first and lacked the clue (its watchers), and their later solves; each such solve pays the contester `preventionBonus`, 0.5 halved every `halfLife` turns of delay. `contestOptions` reads the options from a world. `sharePenalty` is the all-sharing penalty at a board's end, from the warm start's answers per board type. `Records` is each agent's own record of payoffs over a sequence of 6 boards (arm 4).

In [ ]:
#| default_exp contest

In [ ]:
#| hide
from nbdev.showdoc import *

## Contests

In [ ]:
#| export
from dataclasses import dataclass, field

BONUS = 0.5                                   # the evaluation's wrong-answer cost: a prevented wrong answer's price
SLOTS = ('guards', 'all')


def preventionBonus(lag, H):
    "0.5 * 2 ** (-lag / H): the bonus per watcher of a contest made `lag` turns after the lie it answers."
    return BONUS * 2 ** (-float(lag) / float(H))


@dataclass
class Contest:
    t_c: float                                # time of the contest
    who: int                                  # the contester
    k: int                                    # the slot: subtask k, clue j
    j: int
    t_l: float                                # first post of the lie it answers
    lie: int                                  # that lie's residue
    liar: int                                 # its first poster
    watchers: dict = field(default_factory=dict)   # {agent: time of its correct solve of k, or None}


class ContestLog:
    """Contests on one board, built from the shares and what each agent was shown, as the game runs. A share by an honest
    agent of a clue's true value while the slot (k, j) shows a lie (a false value first posted by a liar) is a contest; its
    watchers are the honest agents, other than the contester and not holding clue j, that were shown a lie of the slot before
    any true value of it and before the contest. A watcher's later correct solve of k pays the contester, once per (contest,
    watcher), when the contester is in `contestSlots`."""
    def __init__(self, board, liarSlots, contestSlots, halfLife=4.0):
        self.board, self.liars, self.slots, self.H = board, frozenset(liarSlots), frozenset(contestSlots), float(halfLife)
        self.first, self.seenTrue, self.sawLie = {}, set(), {}   # {(k, j): {r: (poster, t)}}; {(i, k, j)}; {(k, j): {i: t}}
        self.contests, self.paid = [], []                          # paid: (t, contester, watcher, k, j, bonus)

    def _true(self, k, j):
        return self.board.tasks[k].task.clue(j)[1]

    def lies(self, k, j):
        "[(t, r, liar)]: the slot's false values first posted by a liar, oldest first."
        tr = self._true(k, j)
        return sorted((t, r, a) for r, (a, t) in self.first.get((k, j), {}).items() if a in self.liars and r != tr)

    def shownOn(self, visible):
        "{(k, j): residues}: the posted values an agent seeing the subtasks `visible` is shown."
        vis = set(visible)
        return {(k, j): set(v) for (k, j), v in self.first.items() if k in vis}

    def onShown(self, t, i, shown):
        "Agent i was shown `shown` ({(k, j): residues}) at t: note true values seen, and lies seen before any true value."
        if i in self.liars:
            return
        for (k, j), rs in shown.items():
            if (i, k, j) in self.seenTrue:
                continue
            if self._true(k, j) in rs:
                self.seenTrue.add((i, k, j))
            elif i not in self.board.tasks[k].holders[j] and any(r in rs for _, r, _ in self.lies(k, j)):
                self.sawLie.setdefault((k, j), {}).setdefault(i, float(t))

    def onShare(self, t, i, k, j, r, true):
        "Agent i posts residue r (true or not) for clue (k, j) at t; a contest is judged at this moment, before the post counts."
        lies = self.lies(k, j)
        if true and lies and i not in self.liars:
            w = {a: None for a in self.sawLie.get((k, j), {}) if a != i and (a, k, j) not in self.seenTrue}
            self.contests.append(Contest(float(t), i, k, j, lies[0][0], lies[0][1], lies[0][2], w))
        self.first.setdefault((k, j), {}).setdefault(int(r), (i, float(t)))
        if true:                                  # a poster of the true value knows it: no later contest's watcher
            self.seenTrue.add((i, k, j))

    def onSolve(self, t, i, k):
        "A correct solve of k by i at t, strictly after the contest (t > t_c) -> [(contester, bonus)] for the contests it qualifies for, contesters in contestSlots."
        out = []
        for c in self.contests:
            if c.k == k and i in c.watchers and c.watchers[i] is None and t > c.t_c:
                c.watchers[i] = float(t)
                if c.who in self.slots:
                    out.append((c.who, self.bonus(c)))
                    self.paid.append((float(t), c.who, i, k, c.j, out[-1][1]))
        return out

    def bonus(self, c, H=None):
        return preventionBonus(c.t_c - c.t_l, self.H if H is None else H)

    def earned(self, n):
        "Bonus per agent (length n) paid so far."
        out = [0.0] * n
        for _, a, _, _, _, x in self.paid:
            out[a] += x
        return out


def contestOptions(world, N, nLiar):
    "ClueBoardEnv's `contest` options from a world: liars in the first nLiar slots; contestSlots 'guards' or 'all'; halfLife."
    s, g = world.get('contestSlots'), int(world.get('guards', 0))
    if s not in SLOTS:
        raise ValueError(f"scheme 'contest' needs contestSlots in {SLOTS}, got {s!r}")
    slots = range(nLiar, nLiar + g) if s == 'guards' else range(nLiar, N)
    return dict(liarSlots=range(nLiar), contestSlots=slots, halfLife=float(world.get('halfLife', 4.0)))

## The all-sharing penalty

In [ ]:
#| export
def boardType(N, nLiar, life):
    "The key of a board type in `warmAnswers`: (team size, liar count, subtask lifetime)."
    return int(N), int(nLiar), int(life)


def _table(warmAnswers):
    items = warmAnswers.items() if isinstance(warmAnswers, dict) else warmAnswers
    return {tuple(int(x) for x in k): float(v) for k, v in items}


def sharePenalty(answers, boardType, warmAnswers, amount):
    "amount for every honest slot if answers < warmAnswers[boardType] / 3, else 0 (a board type missing from the table fails)."
    ref = _table(warmAnswers)[tuple(int(x) for x in boardType)]
    return float(amount) if answers < ref / 3 else 0.0


def honestAnswers(board, nLiar):
    "Submissions by the honest slots (every slot from nLiar on), whatever their result."
    return sum(1 for s in board.submits if s[1] >= nLiar)


def applySharePenalty(board, nLiar, warmAnswers, amount):
    "At the board's end: the penalty, if due, taken from every honest slot's pending reward (its last reward). -> the penalty."
    p = sharePenalty(honestAnswers(board, nLiar), boardType(board.nAgents, nLiar, board.life), warmAnswers, amount)
    board.endPenalty(range(nLiar, board.nAgents), p)
    return p

## Per-agent records

In [ ]:
#| export
class Records:
    """Each agent's own payoffs from solves and contests over the earlier boards of a sequence of `length` boards (arm 4).
    `addBoard` adds a finished board and starts the next; after `length` boards the records reset (a new sequence)."""
    KINDS = ('answers', 'contests')

    def __init__(self, N, length=6):
        self.N, self.length = int(N), int(length)
        self.reset()

    def reset(self):
        self.boards, self.cur = [], self._fresh()

    def _fresh(self):
        return [dict.fromkeys(self.KINDS, 0.0) for _ in range(self.N)]

    def add(self, i, kind, payoff):
        self.cur[i][kind] += float(payoff)

    def endBoard(self):
        self.boards.append(self.cur)
        self.cur = self._fresh()
        if len(self.boards) >= self.length:
            self.reset()

    def addBoard(self, board, slots):
        "A finished board: per slot, the value of its own answers (solved values minus wrong-answer costs) and its contest bonuses."
        net, log = board.netByAgent(), getattr(board, 'contestLog', None)
        won = log.earned(board.nAgents) if log is not None else [0.0] * board.nAgents
        for i in slots:
            self.add(i, 'answers', net[i])
            self.add(i, 'contests', won[i])
        self.endBoard()

    def line(self, i):
        "The prompt line for agent i: its own record only."
        if not self.boards:
            return 'Your record: this is the first board of a sequence of %d.' % self.length
        parts = [f"board {n + 1}: your answers earned {b[i]['answers']:.2f}, your contests earned {b[i]['contests']:.2f}"
                 for n, b in enumerate(self.boards)]
        return f'Your record on the earlier boards of this sequence of {self.length}: ' + '; '.join(parts) + '.'

## Tests

In [ ]:
import hashlib
import json

import numpy as np

from explRL.clueboard import ClueBoardEnv
from explRL.contest import *

# Hand-built boards: liar slot 0; G a contester, A another honest poster, W a watcher that lacks clue j of subtask k
kw = dict(m=2, r=1, R=30)
b0 = ClueBoardEnv(3, 6, **kw)
k, j = b0.open[0], 0
p, rt = b0.tasks[k].task.clue(j)
lie, x = (rt + 1) % p, b0.tasks[k].task.x
H0 = b0.tasks[k].holders[j]
W = next(a for a in range(1, 6) if a not in H0)
G, A, V = [a for a in range(1, 6) if a != W][:3]          # V: another agent, holding j or not


def play(events, scheme='contest', slots=(G,), H=4.0, N=6):
    b = ClueBoardEnv(3, N, scheme=scheme, contest=dict(liarSlots=[0], contestSlots=slots, halfLife=H), **kw)
    for t, i, a in events:
        b.apply(i, a, float(t))
    return b


share = lambda r: ('share', (k, j, r))
solve = lambda t: [(t, W, ('open', k)), (t + 1, W, ('submit', x))]
gain = lambda ev, a=G, **o: float(play(ev, **o).pending[a] - play(ev, 'credit', **o).pending[a])

In [ ]:
# test_contest (AC-3): a contest at lag 0 and lag H pays 0.5 and 0.25 per qualifying solve, on top of credit's share
for L, want in ((0, 0.5), (4, 0.25), (2, 0.5 * 2 ** -0.5)):
    ev = [(0, 0, share(lie)), (0, W, ('wait', None)), (L, G, share(rt))] + solve(L + 1)
    assert abs(gain(ev) - want) < 1e-12, (L, gain(ev))
    b = play(ev)
    c, = b.contestLog.contests
    assert (c.t_c, c.who, c.k, c.j, c.t_l, c.lie, c.liar) == (L, G, k, j, 0.0, lie, 0) and c.watchers == {W: L + 2.0}
    assert b.contestLog.paid == [(L + 2.0, G, W, k, j, want)] and b.contestLog.earned(6)[G] == want
    assert G in b._creditTo(W, k) and play(ev, 'credit').pending[G] > 0      # the credit share is credit's own rule
assert abs(gain([(0, 0, share(lie)), (0, W, ('wait', None)), (3, G, share(rt))] + solve(4), H=2.0) - 0.5 * 2 ** -1.5) < 1e-12
# a contest against an honest false value pays nothing (and is no contest), even when a liar repeats that value
ev = [(0, A, share(lie)), (0, W, ('wait', None)), (1, 0, share(lie)), (2, G, share(rt))] + solve(3)
assert gain(ev) == 0 and play(ev).contestLog.contests == []
# a non-contest slot (contestSlots 'guards' and G no guard) is paid nothing, though its contest is recorded
ev = [(0, 0, share(lie)), (0, W, ('wait', None)), (1, G, share(rt))] + solve(2)
assert gain(ev, slots=()) == 0 and len(play(ev, slots=()).contestLog.contests) == 1 and play(ev, slots=()).contestLog.paid == []
# an agent that holds the clue does not qualify
h = next(iter(H0)) if 0 not in H0 else None
if h is not None and h != G:
    ev = [(0, 0, share(lie)), (0, h, ('wait', None)), (1, G, share(rt)), (2, h, ('open', k)), (3, h, ('submit', x))]
    assert gain(ev) == 0 and play(ev).contestLog.contests[0].watchers == {}
bh = play([(0, 0, share(lie))], N=6)
holder = next(iter(bh.tasks[k].holders[j]))
bh.contestLog.onShown(1.0, holder, bh.contestLog.shownOn(bh.open))
assert (k, j) not in bh.contestLog.sawLie or holder not in bh.contestLog.sawLie[(k, j)]
# the contester is never its own watcher; a liar's true post is no contest; a solve pays once per (contest, watcher)
ev = [(0, 0, share(lie)), (0, G, ('wait', None)), (1, G, share(rt)), (2, G, ('open', k)), (3, G, ('submit', x))]
assert play(ev).contestLog.contests[0].watchers == {} and play(ev).contestLog.paid == []
assert play([(0, 0, share(lie)), (0, W, ('wait', None)), (1, 0, share(rt))]).contestLog.contests == []
b = play([(0, 0, share(lie)), (0, W, ('wait', None)), (1, G, share(rt))] + solve(2))
assert b.contestLog.onSolve(5.0, W, k) == [] and len(b.contestLog.paid) == 1
# two contests of one lie: each pays its own decayed bonus for a watcher shown the lie before it
ev = [(0, 0, share(lie)), (0, W, ('wait', None)), (1, G, share(rt)), (2, A, share(rt))] + solve(3)
b = play(ev, slots=(G, A))
assert [(c.who, c.watchers) for c in b.contestLog.contests] == [(G, {W: 4.0}), (A, {W: 4.0})]
assert abs(gain(ev, slots=(G, A)) - 0.5 * 2 ** -0.25) < 1e-12 and abs(gain(ev, A, slots=(G, A)) - 0.5 * 2 ** -0.5) < 1e-12
assert abs(preventionBonus(4, 4) - 0.25) < 1e-15 and preventionBonus(0, 4) == 0.5
# a watcher's solve at the contest's own time does not pay (t > t_c, design-low-level); one turn later it does
b = play([(0, 0, share(lie)), (0, W, ('wait', None)), (1, G, share(rt))])
assert b.contestLog.onSolve(1.0, W, k) == [] and b.contestLog.contests[0].watchers == {W: None}
assert b.contestLog.onSolve(1.5, W, k) == [(G, 0.5 * 2 ** -0.25)]

In [ ]:
# test_watchers (AC-3b): a watcher shown a true value before the lie does not qualify (nor one shown both at once)
ev = [(0, A, share(rt)), (1, W, ('wait', None)), (2, 0, share(lie)), (3, W, ('wait', None)), (3, V, ('wait', None)),
      (4, G, share(rt))] + solve(5)
b = play(ev)
assert len(b.contestLog.contests) == 1 and b.contestLog.contests[0].watchers == {} and gain(ev) == 0
# a contest that is not the first true post earns the bonus but not the credit share (the first true poster keeps it)
ev = [(0, 0, share(lie)), (0, W, ('wait', None)), (1, A, share(rt)), (2, G, share(rt))] + solve(3)
bc, bk = play(ev), play(ev, 'credit')
assert bk.pending[G] == 0 and abs(bc.pending[G] - 0.5 * 2 ** -0.5) < 1e-12
assert bc.pending[A] == bk.pending[A] > 0 and bk._creditTo(W, k) == [A]
assert [c.who for c in bc.contestLog.contests] == [A, G] and [x[1] for x in bc.contestLog.paid] == [G]

In [ ]:
# test_credit_equal (AC-2): scheme 'credit' (and 'team', 'individual') rewards equal the code before Phase 19, on rule-agent
# games (boards with and without colony options, guards, squads) and on rollouts with fake replies; digests recorded at d65634c
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import rollout
from explRL.clue_runs import clueSetup
from explRL.colony_agents import faultClass, memberClass, scriptedFaults
from explRL.colony_board import stage4Members
from explRL.llm_agent import runTurns
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import LLMConfig
from explRL.multi import spawnRngs

digest = lambda out: hashlib.sha256(json.dumps(out, sort_keys=True).encode()).hexdigest()[:16]
GAMES = ((4, 0, {}), (8, 2, dict(r=2, colony=True)), (8, 2, dict(r=2, colony=True, guards=2)),
         (16, 4, dict(r=2, colony=True, guards=4, squads=4)))


def ruleGame(sch, N, nl, w, seed, slots=None):
    cfg = LLMConfig(task='clue', channel='blackboard', N=N, turns=30, nLiar=nl, world=w)
    b, ch = clueSetup(cfg, seed)
    b.scheme = sch
    if slots is not None:
        b.contestLog = ContestLog(b, range(nl), slots)
    rngs = spawnRngs(seed + 1000, N)
    runTurns(stage4Members(w, [faultClass(w)(ch, rngs[i]) if i < nl else memberClass(w, i, nl)(ch, rngs[i]) for i in range(N)]), b, 30)
    return b


out = []
for sch in ('credit', 'team', 'individual'):
    for N, nl, w in GAMES:
        for seed in (900, 901):
            b = ruleGame(sch, N, nl, w, seed)
            out.append((sch, N, nl, seed, [round(r[7], 12) for r in b.ledger], np.round(b.pending, 12).tolist(), b.counts))
assert digest(out) == '64676b4bf4fb098b', digest(out)
fake = lambda: LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
out = []
for sch in ('credit', 'team', 'individual'):
    for N, nl, w in ((4, 0, dict(m=2, R=30)), (4, 1, dict(m=2, R=30)), (8, 2, dict(r=2, colony=True)), (8, 2, dict(r=2, colony=True, guards=2))):
        cfg = LLMConfig(task='clue', channel='blackboard', N=N, turns=10, nLiar=nl, world=dict(w, scheme=sch))
        ro = rollout(cfg, [900, 901], claudePolicy(fake()), scriptedFaults(cfg.world, nl))
        out.append((sch, N, nl, [[[x['reward'] for x in lg] for lg in lgs] for lgs in ro.logs], np.round(ro.returns, 12).tolist(),
                    [bd.counts for bd in ro.boards]))
assert digest(out) == 'b75e5d2a6fdbff55', digest(out)
# 'contest' with no contest slot is 'credit' exactly; with every honest slot, rewards differ by exactly the bonuses paid
for N, nl, w in GAMES[1:]:
    for seed in (900, 901):
        cr, none, every = ruleGame('credit', N, nl, w, seed), ruleGame('contest', N, nl, w, seed, ()), ruleGame('contest', N, nl, w, seed, range(nl, N))
        assert [r[7] for r in none.ledger] == [r[7] for r in cr.ledger] and (none.pending == cr.pending).all()
        tot = lambda b: np.bincount([r[0] for r in b.ledger], [r[7] for r in b.ledger], N) + b.pending
        assert np.allclose(tot(every) - tot(cr), every.contestLog.earned(N)) and every.counts == cr.counts
        assert all(nl <= a < N for _, a, *_ in every.contestLog.paid)
assert sum(len(ruleGame('contest', 16, 4, GAMES[3][2], s, range(4, 16)).contestLog.paid) for s in (900, 901)) > 0

In [ ]:
# test_penalty (AC-4, penalty part): the penalty fires exactly when answers fall below a third of the reference, on every
# honest slot; the table may be a dict or (key, value) pairs (JSON lists); a board type missing from it fails
T = {(4, 1, 30): 9.0}
assert sharePenalty(2, (4, 1, 30), T, 0.5) == 0.5 and sharePenalty(3, (4, 1, 30), T, 0.5) == 0.0
assert sharePenalty(2, [4, 1, 30.0], [([4, 1, 30], 9)], 0.5) == 0.5 and boardType(4, 1, 30.0) == (4, 1, 30)
try:
    sharePenalty(0, (8, 2, 30), T, 0.5)
    raise AssertionError('missing board type')
except KeyError:
    pass
b = ClueBoardEnv(3, 4, **kw)
x4 = b.tasks[k].task.x
b.apply(2, ('open', k), 0.0)
b.apply(2, ('submit', (x4 + 1) % 30), 1.0)                # one honest answer (wrong); slot 0's answer does not count
b.apply(0, ('open', k), 1.0)
b.apply(0, ('submit', x4), 2.0)
assert honestAnswers(b, 1) == 1
before = b.pending.copy()
assert applySharePenalty(b, 1, {(4, 1, 30): 3.0}, 0.5) == 0.0 and (b.pending == before).all()    # 1 == 3 / 3: not below
assert applySharePenalty(b, 1, {(4, 1, 30): 3.5}, 0.5) == 0.5 and (b.pending - before).tolist() == [0, -0.5, -0.5, -0.5]
# through rollout: world sharePenalty and warmAnswers take it from every honest slot's return, and from its last reward-to-go
cfg = lambda **o: LLMConfig(task='clue', channel='blackboard', N=4, turns=6, nLiar=1, world=dict(m=2, R=30, scheme='credit', **o))
ref = rollout(cfg(), [900], claudePolicy(fake()), scriptedFaults({}, 1))
pen = rollout(cfg(sharePenalty=0.5, warmAnswers={(4, 1, 30): 1e6}), [900], claudePolicy(fake()), scriptedFaults({}, 1))
off = rollout(cfg(sharePenalty=0.5, warmAnswers={(4, 1, 30): 0.0}), [900], claudePolicy(fake()), scriptedFaults({}, 1))
assert np.allclose(pen.returns - ref.returns, [[0, -0.5, -0.5, -0.5]]) and (off.returns == ref.returns).all()
assert (pen.boards[0].pending - ref.boards[0].pending).tolist() == [0, -0.5, -0.5, -0.5]

In [ ]:
# test_records (AC-4b): an agent's record lists only its own payoffs, and resets between sequences of 6 boards
r = Records(3)
assert r.line(1) == 'Your record: this is the first board of a sequence of 6.'
for n in range(5):
    r.add(1, 'answers', 1.25 + n)
    r.add(2, 'contests', 7.5)
    r.endBoard()
assert '1.25' in r.line(1) and '7.50' not in r.line(1) and '7.50' in r.line(2) and '1.25' not in r.line(2)
assert r.line(1).count('board ') == 5 and 'board 5: your answers earned 5.25, your contests earned 0.00' in r.line(1)
r.endBoard()                                               # the sixth board ends the sequence
assert r.boards == [] and r.line(1).startswith('Your record: this is the first')
# through rollout: the honest prompts end with the agent's own line, and each board's payoffs are added to its record
c = LLMConfig(task='clue', channel='blackboard', N=4, turns=4, nLiar=1, world=dict(m=2, R=30, colony=True, guards=1, scheme='contest', contestSlots='all'))
recs = [Records(4), Records(4)]
for n in range(2):
    ro = rollout(c, [900, 901], claudePolicy(fake()), scriptedFaults({}, 1), keepPrompts=True, records=recs)
    for b in range(2):
        net, won = ro.boards[b].netByAgent(), ro.boards[b].contestLog.earned(4)
        assert [(x['answers'], x['contests']) for x in recs[b].boards[n]] == [(0, 0)] + [(net[i], won[i]) for i in (1, 2, 3)]
        for i in (1, 2, 3):
            assert all(p[1][-1]['text'].endswith(('Your record: this is the first board of a sequence of 6.' if n == 0 else
                                                  f"board 1: your answers earned {recs[b].boards[0][i]['answers']:.2f}, your contests earned {recs[b].boards[0][i]['contests']:.2f}.")) for p in ro.prompts[b][i])
assert len(recs[0].boards) == 2
# contest options: contestSlots must be 'guards' or 'all'; the blackboard channel only
assert list(contestOptions(dict(guards=2, contestSlots='guards'), 8, 2)['contestSlots']) == [2, 3]
assert list(contestOptions(dict(contestSlots='all', halfLife=2), 8, 2)['contestSlots']) == list(range(2, 8))
for bad in (dict(c.world, contestSlots=None), dict(c.world, contestSlots='guard')):
    try:
        rollout(LLMConfig(task='clue', channel='blackboard', N=4, turns=1, nLiar=1, world=bad), [900], claudePolicy(fake()))
        raise AssertionError(bad)
    except ValueError:
        pass
try:
    rollout(LLMConfig(task='clue', channel='messages', N=4, turns=1, world=c.world), [900], claudePolicy(fake()))
    raise AssertionError('messages')
except ValueError:
    pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()